In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Generate synthetic data
np.random.seed(42)

num_samples = 1000

# Features: duration, protocol_type (numeric), service (numeric), flag (numeric), src_bytes, dst_bytes
# Update the data dictionary to ensure float types for the math later
data = {
    'duration': np.random.uniform(0.1, 1000, num_samples).astype(float),
    'protocol_type': np.random.randint(0, 3, num_samples),
    'service': np.random.randint(0, 10, num_samples),
    'flag': np.random.randint(0, 10, num_samples),
    'src_bytes': np.random.randint(0, 100000, num_samples).astype(float),
    'dst_bytes': np.random.randint(0, 50000, num_samples).astype(float),
    'is_malicious': np.random.randint(0, 2, num_samples)
}

df = pd.DataFrame(data)

# Introduce some correlation for malicious samples
df.loc[df['is_malicious'] == 1, 'src_bytes'] = df.loc[df['is_malicious'] == 1, 'src_bytes'] * np.random.uniform(1.5, 5.0)
df.loc[df['is_malicious'] == 1, 'dst_bytes'] = df.loc[df['is_malicious'] == 1, 'dst_bytes'] * np.random.uniform(0.5, 2.0)
df.loc[df['is_malicious'] == 1, 'duration'] = df.loc[df['is_malicious'] == 1, 'duration'] * np.random.uniform(1.2, 3.0)

# Ensure target distribution is not perfectly balanced
malicious_count = df['is_malicious'].sum()
normal_count = len(df) - malicious_count
print(f"Total samples: {len(df)}")
print(f"Normal samples: {normal_count}")
print(f"Malicious samples: {malicious_count}")

# Define features (X) and target (y)
X = df.drop('is_malicious', axis=1)
y = df['is_malicious']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nData preparation complete.")
print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")

Total samples: 1000
Normal samples: 494
Malicious samples: 506

Data preparation complete.
Training set shape: (700, 6)
Testing set shape: (300, 6)


In [ ]:
training the model

In [3]:
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression model
# Using default solver, but 'liblinear' is good for smaller datasets
# Increased max_iter to ensure convergence for potentially complex data
model = LogisticRegression(random_state=42, max_iter=1000)

# Train the model
model.fit(X_train_scaled, y_train)

print("\nLogistic Regression model trained successfully.")
print(f"Model coefficients: {model.coef_}")
print(f"Model intercept: {model.intercept_}")


Logistic Regression model trained successfully.
Model coefficients: [[ 1.91852683 -0.07086802 -0.07258895  0.01279148  1.25061707  1.02631827]]
Model intercept: [0.50880118]


In [ ]:
testing the model

In [4]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict on the test set
y_pred = model.predict(X_test_scaled)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(class_report)

# Interpretation of Metrics:
# Accuracy: Overall correctness of the model.
# Precision: Of the predicted malicious instances, how many were actually malicious?
# Recall: Of the actual malicious instances, how many did the model correctly identify?
# F1-Score: Harmonic mean of precision and recall.
# Confusion Matrix:
# [[True Negatives (TN), False Positives (FP)]]
# [[False Negatives (FN), True Positives (TP)]]

# Example: Predicting a new instance
# Let's assume a new connection with the same features as the first training sample
new_instance_scaled = X_test_scaled[0].reshape(1, -1)
prediction = model.predict(new_instance_scaled)
probability = model.predict_proba(new_instance_scaled)

print(f"\nPrediction for the first test instance: {'Malicious' if prediction[0] == 1 else 'Normal'}")
print(f"Probability (Normal, Malicious): {probability[0]}")


Model Accuracy: 0.8700

Confusion Matrix:
[[134  14]
 [ 25 127]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.91      0.87       148
           1       0.90      0.84      0.87       152

    accuracy                           0.87       300
   macro avg       0.87      0.87      0.87       300
weighted avg       0.87      0.87      0.87       300


Prediction for the first test instance: Malicious
Probability (Normal, Malicious): [0.04087034 0.95912966]
